In [2]:
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()

te = data['test']
print("=== test split fields ===")
for k, v in te.items():
    arr = np.asarray(v)
    head = arr[:8] if arr.ndim == 1 else f"ndim={arr.ndim}, shape={arr.shape}"
    print(f"{k:10s} shape={str(arr.shape):18s} dtype={str(arr.dtype):10s} first: {head}")

for k in te:
    if 'pat' in k.lower():
        print("\nunique patients:", np.unique(te[k]))

Mounted at /content/drive
=== test split fields ===
X          shape=(522, 18, 2560)    dtype=float32    first: ndim=3, shape=(522, 18, 2560)
y          shape=(522,)             dtype=int64      first: [1 0 0 0 0 0 0 1]
patient    shape=(522,)             dtype=<U5        first: ['chb20' 'chb24' 'chb23' 'chb24' 'chb23' 'chb23' 'chb24' 'chb23']
file       shape=(522,)             dtype=<U12       first: ['chb20_14.edf' 'chb24_05.edf' 'chb23_10.edf' 'chb24_02.edf'
 'chb23_20.edf' 'chb23_16.edf' 'chb24_13.edf' 'chb23_09.edf']
t0         shape=(522,)             dtype=int64      first: [ 1990  1430  4290  3590   730 13250    40  2610]

unique patients: ['chb20' 'chb22' 'chb23' 'chb24']


In [3]:
import numpy as np

te  = data['test']
y   = te['y'].astype(int)
t0  = te['t0'].astype(int)
fil = np.asarray(te['file'])

order = np.lexsort((t0, fil))
y, t0, fil = y[order], t0[order], fil[order]

all_gaps = []
for f in np.unique(fil):
    tt = np.sort(t0[fil == f])
    if len(tt) > 1:
        all_gaps.extend(np.diff(tt).tolist())
all_gaps = np.array(all_gaps)
print("min gap between windows in a file:", all_gaps.min(), "s  (this is the stride)")
print("gap percentiles 50/90/99 (s):", np.percentile(all_gaps, [50, 90, 99]).round().tolist())

def count_events(gap_thresh):
    total, per_pat = 0, {}
    for f in np.unique(fil):
        tt = np.sort(t0[(fil == f) & (y == 1)])
        if len(tt) == 0:
            continue
        n_ev = 1 + int(np.sum(np.diff(tt) > gap_thresh))
        total += n_ev
        p = f.split('_')[0]
        per_pat[p] = per_pat.get(p, 0) + n_ev
    return total, per_pat

for thr in [10, 20, 30, 60]:
    tot, pp = count_events(thr)
    print(f"merge gap <= {thr:3d}s  ->  {tot:3d} events   {pp}")

n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
stride = int(all_gaps.min())
print(f"\npositive windows: {n_pos}   negative windows: {n_neg}")
print(f"if stride={stride}s, saved interictal span = {n_neg*stride/3600:.2f} h "
      f"(subsampled, not true recording hours)")

min gap between windows in a file: 10 s  (this is the stride)
gap percentiles 50/90/99 (s): [210.0, 1256.0, 3081.0]
merge gap <=  10s  ->   34 events   {'chb20': 8, 'chb22': 3, 'chb23': 7, 'chb24': 16}
merge gap <=  20s  ->   34 events   {'chb20': 8, 'chb22': 3, 'chb23': 7, 'chb24': 16}
merge gap <=  30s  ->   34 events   {'chb20': 8, 'chb22': 3, 'chb23': 7, 'chb24': 16}
merge gap <=  60s  ->   34 events   {'chb20': 8, 'chb22': 3, 'chb23': 7, 'chb24': 16}

positive windows: 174   negative windows: 348
if stride=10s, saved interictal span = 0.97 h (subsampled, not true recording hours)


In [4]:
import numpy as np
from sklearn.linear_model import RidgeClassifier

SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()

y_tr = data['train']['y'].astype(int)
te   = data['test']
y_te = te['y'].astype(int)
file = np.asarray(te['file'])
t0   = te['t0'].astype(int)

MERGE = 30
event_id = np.full(len(y_te), -1, dtype=int)
next_ev = 0
for f in np.unique(file):
    idx = np.where((file == f) & (y_te == 1))[0]
    if len(idx) == 0:
        continue
    order = idx[np.argsort(t0[idx])]
    starts = t0[order]
    ev_local = np.zeros(len(order), dtype=int)
    for i in range(1, len(order)):
        ev_local[i] = ev_local[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
    for local, gi in zip(ev_local, order):
        event_id[gi] = next_ev + local
    next_ev += int(ev_local.max()) + 1

n_events = next_ev
neg_mask = (y_te == 0)
print("ground-truth seizure events:", n_events, "| interictal windows:", int(neg_mask.sum()))

def sens_at_fpr(y_prob, fpr_targets=(0.05, 0.10, 0.20)):
    rows = []
    for thr in np.unique(y_prob):
        pred = y_prob >= thr
        det = np.array([pred[event_id == ev].any() for ev in range(n_events)])
        rows.append((thr, det.mean(), pred[neg_mask].mean()))
    rows = np.array(rows)
    out = {}
    for tgt in fpr_targets:
        ok = rows[rows[:, 2] <= tgt]
        out[tgt] = ok[:, 1].max() if len(ok) else np.nan
    return out, rows

Ztr = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_train_emb.npy')
Zte = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_test_emb.npy')
clf = RidgeClassifier().fit(Ztr, y_tr)
y_prob = clf.decision_function(Zte)

res, curve = sens_at_fpr(y_prob)
print("\nChronos-2 event-level sensitivity:")
for tgt, s in res.items():
    print(f"  at {int(tgt*100)}% interictal FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events} events)")

ground-truth seizure events: 34 | interictal windows: 348

Chronos-2 event-level sensitivity:
  at 5% interictal FPR: sensitivity = 0.882  (30/34 events)
  at 10% interictal FPR: sensitivity = 0.971  (33/34 events)
  at 20% interictal FPR: sensitivity = 1.000  (34/34 events)


In [5]:
Ztr_m = np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_train_emb.npy')
Zte_m = np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_test_emb.npy')
clf_m = RidgeClassifier().fit(Ztr_m, y_tr)
prob_m = clf_m.decision_function(Zte_m)

res_m, _ = sens_at_fpr(prob_m)
print("MantisV2 event-level sensitivity:")
for tgt, s in res_m.items():
    print(f"  at {int(tgt*100)}% interictal FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events} events)")

/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.43729e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


MantisV2 event-level sensitivity:
  at 5% interictal FPR: sensitivity = 0.794  (27/34 events)
  at 10% interictal FPR: sensitivity = 0.882  (30/34 events)
  at 20% interictal FPR: sensitivity = 0.941  (32/34 events)


In [6]:
fs = 256
bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 64)}

def band_power(X):
    T = X.shape[2]
    freqs = np.fft.rfftfreq(T, 1/fs)
    psd = np.abs(np.fft.rfft(X, axis=2))**2
    feats = [np.log(psd[:, :, (freqs >= lo) & (freqs < hi)].sum(axis=2) + 1e-8)
             for lo, hi in bands.values()]
    return np.concatenate(feats, axis=1)

def corr_feats(X):
    C = X.shape[1]
    iu = np.triu_indices(C, 1)
    out = np.empty((X.shape[0], len(iu[0])))
    for i in range(X.shape[0]):
        out[i] = np.corrcoef(X[i])[iu]
    return np.nan_to_num(out)

Xtr, Xte = data['train']['X'], data['test']['X']
Ftr = np.concatenate([band_power(Xtr), corr_feats(Xtr)], axis=1)
Fte = np.concatenate([band_power(Xte), corr_feats(Xte)], axis=1)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sc = StandardScaler().fit(Ftr)
clf_c = LogisticRegression(max_iter=2000).fit(sc.transform(Ftr), y_tr)
prob_c = clf_c.decision_function(sc.transform(Fte))

print("classical band+corr accuracy on v2:", round(accuracy_score(y_te, clf_c.predict(sc.transform(Fte))), 4))
res_c, _ = sens_at_fpr(prob_c)
print("Classical (band+corr) event-level sensitivity:")
for tgt, s in res_c.items():
    print(f"  at {int(tgt*100)}% interictal FPR: sensitivity = {s:.3f}  ({int(round(s*n_events))}/{n_events} events)")

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


classical band+corr accuracy on v2: 0.8697
Classical (band+corr) event-level sensitivity:
  at 5% interictal FPR: sensitivity = 0.912  (31/34 events)
  at 10% interictal FPR: sensitivity = 0.971  (33/34 events)
  at 20% interictal FPR: sensitivity = 1.000  (34/34 events)


In [8]:
import subprocess
print(subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*result*.xlsx'],
                      capture_output=True, text=True).stdout)

/content/drive/MyDrive/results.xlsx



In [10]:
from openpyxl import load_workbook
from openpyxl.styles import Font

XLSX = "/content/drive/MyDrive/results.xlsx"

wb = load_workbook(XLSX)
bold = Font(bold=True)
sheet_name = "RQ3 event sensitivity"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

ws.cell(row=1, column=1, value="RQ3: event-level sensitivity at fixed interictal FPR (CHB-MIT v2 test, 34 events)").font = bold

headers = ["Method", "Channel handling", "Sens @5% FPR", "Sens @10% FPR", "Sens @20% FPR"]
for c, h in enumerate(headers, 1):
    ws.cell(row=3, column=c, value=h).font = bold

results_table = [
    ("Chronos-2", "channel-aware (group attention)", res[0.05], res[0.10], res[0.20]),
    ("MantisV2", "channel-independent", res_m[0.05], res_m[0.10], res_m[0.20]),
    ("Classical band+corr", "explicit cross-channel features", res_c[0.05], res_c[0.10], res_c[0.20]),
]

r = 4
for row in results_table:
    for c, v in enumerate(row, 1):
        cell = ws.cell(row=r, column=c, value=v)
        if c > 2:
            cell.number_format = "0.000"
    r += 1

wb.save(XLSX)
print("saved to", XLSX)

saved to /content/drive/MyDrive/results.xlsx


In [13]:
import subprocess
r1 = subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*rocket*'], capture_output=True, text=True)
r2 = subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*moment*'], capture_output=True, text=True)
print("ROCKET matches:")
print(r1.stdout if r1.stdout.strip() else "(none found)")
print("\nMOMENT matches:")
print(r2.stdout if r2.stdout.strip() else "(none found)")

ROCKET matches:
/content/drive/MyDrive/Colab Notebooks/ROCKET.ipynb
/content/drive/MyDrive/Colab Notebooks/ROCKET2.ipynb
/content/drive/MyDrive/diss-datasets/rocket_all_results.csv
/content/drive/MyDrive/diss-datasets/rocket_channel_penalty_MiniRocket.png
/content/drive/MyDrive/diss-datasets/rocket_channel_penalty_MultiRocket.png
/content/drive/MyDrive/diss-datasets/rocket_summary.json
/content/drive/MyDrive/diss-datasets/rocket_all_results_v2.csv


MOMENT matches:
/content/drive/MyDrive/Colab Notebooks/RunningdatasetsMOMENT.ipynb
/content/drive/MyDrive/MOMENT_SYNTH_trained.pt
/content/drive/MyDrive/MOMENT_CMJ_trained.pt
/content/drive/MyDrive/MOMENT_MP8_trained.pt
/content/drive/MyDrive/MOMENT_MP50_trained.pt



In [14]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/diss-datasets/rocket_all_results_v2.csv')
print(df.columns.tolist())
print(df[df['dataset'].astype(str).str.contains('CHB', case=False, na=False)] if 'dataset' in df.columns else df.head(20))

['accuracy', 'macro_f1', 'seconds', 'num_kernels', 'n_features', 'dataset', 'method', 'variant', 'n_channels', 'risk', 'feature_budget', 'pos_precision', 'pos_recall', 'pos_f1']
    accuracy  macro_f1     seconds  num_kernels  n_features  dataset  \
20  0.888889  0.864355  240.023887        19992       19992  CHB-MIT   
21  0.854406  0.827549  123.783663         1092       19656  CHB-MIT   
22  0.722222  0.700360  495.671497         2520       20160  CHB-MIT   
23  0.657088  0.630099  205.086660          168       24192  CHB-MIT   

         method         variant  n_channels     risk  feature_budget  \
20   MiniRocket    multivariate          18  8463360           20000   
21   MiniRocket  single-channel          18  8463360           20000   
22  MultiRocket    multivariate          18  8463360           20000   
23  MultiRocket  single-channel          18  8463360           20000   

    pos_precision  pos_recall    pos_f1  
20       0.960317    0.695402  0.806667  
21       0.84507